# Selección congelada y reproducción desde Model Registry
El ganador y challenger se eligen por CV antes de abrir el holdout. Se evalúan una sola vez; el resultado de prueba no cambia la elección. Las reejecuciones leen la decisión guardada y verifican el alias, sin elegir de nuevo.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from IPython.display import display, Image
import pandas as pd
from src.datos import DOCS, split
from src.pipeline import make_pipeline, PhysicalFeatures


In [2]:
from src.experimentos import final, reproduce, export_runs
selection=final()
display(pd.DataFrame([{**{k:r[k] for k in ["alias","name","final_run_id","version","threshold","cv_ap_mean","cv_ap_std"]}, **r["test"]} for r in selection["models"]]))
display({"cv_difference":selection["cv_difference"],"exceeds_fold_std":selection["difference_exceeds_fold_std"]})

,alias,name,final_run_id,version,threshold,cv_ap_mean,cv_ap_std,ap,roc_auc,recall,precision,f1,cost_per_1000,alert_rate,tp,fp,fn,tn
0,champion,hist_boosting/config_17,d07d9f7f061848138579716948af2c6f,1,0.090636,0.878781,0.026033,0.909850,0.985264,0.897059,0.628866,0.739394,98000.0,0.0485,61,36,7,1896
1,challenger,random_forest/config_13,1bb7401579664e38af7d1ab7e1bb8ea9,2,0.078452,0.873025,0.033908,0.893324,0.977602,0.911765,0.462687,0.613861,104000.0,0.0670,62,72,6,1860


{'cv_difference': 0.005755921628501914, 'exceeds_fold_std': False}

El costo simulado usa Cp=500, Ce=8000 y Cr=1500 USD, heredados del PSet 2 como supuestos. Se elige umbral con predicciones out of fold del entrenamiento y máximo 12 % de alertas en ese conjunto. La tasa en holdout se reporta; no se garantiza capacidad futura. AP no depende de umbral. Las probabilidades no se declaran calibradas.

In [3]:
verification=reproduce()
display(verification)
assert verification["reproduced"]

{'registry_uri': 'models:/ai4i-mantenimiento-modelo@champion',
 'reproduced': True,
 'metrics': {'ap': 0.9098503380639453,
  'roc_auc': 0.9852636706856656,
  'recall': 0.8970588235294118,
  'precision': 0.6288659793814433,
  'f1': 0.7393939393939394,
  'cost_per_1000': 98000.0,
  'alert_rate': 0.0485,
  'tp': 61,
  'fp': 36,
  'fn': 7,
  'tn': 1896}}

In [4]:
exported=export_runs()
display(exported.groupby("tags.phase").size())
print("docs/corridas.csv actualizado con mlflow.search_runs")

tags.phase
ablation          1
baseline          2
candidato         5
final             2
foundation        1
llm               1
tuning           90
tuning_parent     3
dtype: int64

docs/corridas.csv actualizado con mlflow.search_runs


La reproducción en kernel limpio carga models:/ai4i-mantenimiento-modelo@champion y exige igualdad de métricas con tolerancia 1e-12. Los datos sintéticos no demuestran reducción de paradas ni transferencia a planta. El registro deja un candidato reproducible para PSet 4, no una autorización de despliegue industrial.